# Video to frames

In [ ]:
import cv2
import os

# Define input video path and output folder
video_path = '/content/drive/MyDrive/bike_lane/picoR_crop6.mp4'
output_folder = '/content/drive/MyDrive/bike_lane/frames_picoR_crop6'

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Open the video file
video = cv2.VideoCapture(video_path)

if not video.isOpened():
    print(f"Error: Could not open video file {video_path}")
else:
    frame_count = 0
    while True:
        ret, frame = video.read()  # Read a frame
        if not ret:
            break  # Exit when no frames are left

        # Save frame as an image
        frame_filename = os.path.join(output_folder, f'frame_{frame_count:04d}.jpg')
        cv2.imwrite(frame_filename, frame)
        frame_count += 1

    print(f"Video converted to frames successfully! Total frames: {frame_count}")
    print(f"Frames are saved in: {output_folder}")

# Release the video object
video.release()


Error: Could not open video file /content/drive/MyDrive/bike_lane/picoR_crop6.mp4


# **Code**

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 51.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
import os
import cv2
from ultralytics import YOLO

# ── CONFIG ────────────────────────────────────────────────────────────────
INPUT_FRAMES_DIR  = '/content/drive/MyDrive/bike_lane/frames_picoR_crop3'
OUTPUT_FRAMES_DIR = '/content/output_frames(1)'

# your two lines (already given)
LINE1 = ((42, 379), (441, 351))
LINE2 = ((67, 508), (575, 473))

# distance threshold for 'touching' a line (in pixels)
LINE_THRESH = 5

# Which YOLO classes to watch (0=person, 1=bicycle, 2=car, 3=motorbike, 5=bus, 7=truck...)
#WATCHED_CLASSES = {0, 2, 3, 5, 7}
WATCHED_CLASSES = {0, 1, 2, 3, 5, 7}

# ── SETUP ─────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_FRAMES_DIR, exist_ok=True)
model = YOLO('yolov8n.pt')  # or your preferred checkpoint

# Keep state per track ID: { track_id: {"t1":bool, "violation":bool} }
state = {}

# Utility: point-to-line distance
def point_to_line_dist(pt, line):
    (x0,y0), = [pt]
    (x1,y1),(x2,y2) = line
    num = abs((y2-y1)*x0 - (x2-x1)*y0 + x2*y1 - y2*x1)
    den = ((y2-y1)**2 + (x2-x1)**2)**0.5
    return num/den

def touches(line, center):
    return point_to_line_dist(center, line) < LINE_THRESH

# ── PROCESS ───────────────────────────────────────────────────────────────
# Use YOLOv8's built-in tracker: it reads frames & yields tracked boxes + IDs
results = model.track(
    source=INPUT_FRAMES_DIR + '/*.jpg',
    show=False,
    persist=True,
    save=False,
    stream=True
)

for frame_idx, res in enumerate(results):
    # load the original frame to draw on
    frame_path = sorted(os.listdir(INPUT_FRAMES_DIR))[frame_idx]
    img = cv2.imread(os.path.join(INPUT_FRAMES_DIR, frame_path))

    # `res.boxes` contains x1,y1,x2,y2, confidence, class, and .id
    for box in res.boxes:
        cls = int(box.cls.cpu().item())
        tid = int(box.id.cpu().item())
        if cls not in WATCHED_CLASSES:
            continue

        # compute centroid
        x1,y1,x2,y2 = map(int, box.xyxy.cpu().tolist()[0])
        cx, cy = (x1+x2)//2, (y1+y2)//2

        # init state if new track
        if tid not in state:
            state[tid] = {"t1": False, "violation": False}

        st = state[tid]
        # check sequence
        if not st["t1"] and touches(LINE1, (cx,cy)):
            st["t1"] = True
        if st["t1"] and not st["violation"] and touches(LINE2, (cx,cy)):
            st["violation"] = True

        # if violation ever flagged, draw red box + label
        if st["violation"]:
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,0,255), 2)
            cv2.putText(
                img,
                'Violation',
                (x1, y1-10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0,0,255),
                2
            )

    # save output frame
    out_path = os.path.join(OUTPUT_FRAMES_DIR, frame_path)
    cv2.imwrite(out_path, img)

print("Done – annotated frames in", OUTPUT_FRAMES_DIR)



image 1/1622 /content/drive/MyDrive/bike_lane/frames_picoR_crop3/frame_0000.jpg: 640x640 16 persons, 3 bicycles, 2 cars, 2 motorcycles, 3 trucks, 8.0ms
image 2/1622 /content/drive/MyDrive/bike_lane/frames_picoR_crop3/frame_0001.jpg: 640x640 12 persons, 3 bicycles, 2 cars, 3 motorcycles, 3 trucks, 8.0ms
image 3/1622 /content/drive/MyDrive/bike_lane/frames_picoR_crop3/frame_0002.jpg: 640x640 13 persons, 2 bicycles, 2 cars, 2 motorcycles, 3 trucks, 8.0ms
image 4/1622 /content/drive/MyDrive/bike_lane/frames_picoR_crop3/frame_0003.jpg: 640x640 13 persons, 1 bicycle, 3 cars, 1 motorcycle, 3 trucks, 8.0ms
image 5/1622 /content/drive/MyDrive/bike_lane/frames_picoR_crop3/frame_0004.jpg: 640x640 13 persons, 1 bicycle, 3 cars, 2 motorcycles, 2 trucks, 9.6ms
image 6/1622 /content/drive/MyDrive/bike_lane/frames_picoR_crop3/frame_0005.jpg: 640x640 14 persons, 3 bicycles, 3 cars, 2 trucks, 8.9ms
image 7/1622 /content/drive/MyDrive/bike_lane/frames_picoR_crop3/frame_0006.jpg: 640x640 15 persons, 2 bi

# Frames to video

In [ ]:
import cv2
import os

def create_video_from_frames(frames_folder, output_video_path, fps=30):
    """
    Combine frames from a folder into a video file.

    :param frames_folder: Path to the folder containing frames.
    :param output_video_path: Path to save the output video.
    :param fps: Frames per second for the video.
    """
    # Get the list of frame files
    frame_files = sorted([f for f in os.listdir(frames_folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])

    if not frame_files:
        print("No frames found in the folder.")
        return

    # Read the first frame to get dimensions
    first_frame_path = os.path.join(frames_folder, frame_files[0])
    first_frame = cv2.imread(first_frame_path)
    height, width, _ = first_frame.shape

    # Initialize the video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 videos
    video_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    # Iterate over the frame files and write them to the video
    for frame_file in frame_files:
        frame_path = os.path.join(frames_folder, frame_file)
        frame = cv2.imread(frame_path)
        video_writer.write(frame)

    # Release the video writer
    video_writer.release()
    print(f"Video saved at: {output_video_path}")

# Inputs
frames_folder = '/content/output_frames(1)'  # Folder containing processed frames
output_video_path = '/content/drive/MyDrive/bike_lane/result/picoR_crop3_final.mp4'  # Path for the output video
fps = 30  # Frames per second

# Create the video
create_video_from_frames(frames_folder, output_video_path, fps)


Video saved at: /content/drive/MyDrive/bike_lane/result/picoR_crop3_final.mp4


# **crop large videos**

In [ ]:
import cv2
import os

# Define input and output paths
input_video  = "/content/drive/MyDrive/bike_lane/Pico_road.mp4"
output_video = "/content/drive/MyDrive/bike_lane/picoR_crop6.mp4"

# Define start and end times (in seconds)
start_time = 300   # from the very start
end_time   = 360   # up to the first 60 seconds

# Open video file
cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise IOError(f"Cannot open video {input_video}")

# Get video properties
fps          = cap.get(cv2.CAP_PROP_FPS)
frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Prepare writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(output_video, fourcc, fps, (frame_width, frame_height))

# Compute frame indices
start_frame   = int(start_time * fps)
end_frame     = int(end_time   * fps)
current_frame = 0

# Read through and write frames in the desired window
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if start_frame <= current_frame < end_frame:
        out.write(frame)
    elif current_frame >= end_frame:
        break

    current_frame += 1

# Release resources
cap.release()
out.release()

print(f"Video cropping completed. Saved as {os.path.basename(output_video)}")


Video cropping completed. Saved as picoR_crop6.mp4
